# 01 — Without A2A: The Pain of Custom Agent Integration

## Why this notebook exists

Imagine you have two services that need to collaborate:

- A **researcher** that gathers facts on a topic.
- A **writer** that turns those facts into a paragraph.

In a world without a standard agent-to-agent protocol, the only way to connect them is to design a custom REST API for each service, then hand-roll the glue code. That works — until one side changes its contract. Then the glue breaks, and every other consumer of that service breaks with it.

This notebook reproduces that pain in ~100 lines of code. Once you've felt it, the rest of the series will introduce **A2A** — a standard that makes this problem mostly go away.

## What you'll learn

- How to stand up two FastAPI services in a notebook and call them with `httpx`.
- Why ad-hoc REST contracts between agent-like services don't scale.
- What a "breaking change" looks like in practice, and why coordination is expensive without a shared protocol.
- The motivation for the rest of this series: **A2A** as a standard for agent-to-agent communication.

## 1. Setup

We'll use:

- **FastAPI** to define each service.
- **uvicorn** running on a background thread so we can both serve and call the API from the same notebook.
- **httpx** as our HTTP client.
- **pydantic** for typed request/response models.

The helper `run_server_in_thread(app, port)` starts a uvicorn server in a daemon thread and returns a handle we can use to shut it down at the end of the notebook.

In [ ]:
import socket
import threading
import time

import httpx
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    """Start `app` on localhost:`port` in a background daemon thread.

    Returns the uvicorn Server object so the caller can stop it later.
    """
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")

    _servers.append(server)
    return server


def _port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def stop_server(server: uvicorn.Server, port: int) -> None:
    """Stop a single server and wait for its port to be released."""
    server.should_exit = True
    for _ in range(50):
        if _port_is_free(port):
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Port {port} still bound after stop")
    if server in _servers:
        _servers.remove(server)


def shutdown_all_servers() -> None:
    """Stop every server we started in this notebook."""
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


print("Setup OK")

## 2. The Researcher Service

The researcher takes a `topic` and returns a list of "facts" about it. We're not using a real LLM — this is a stub that returns canned facts, because the point of this notebook is the **integration**, not the intelligence.

The researcher's API contract: `POST /research` with `{"topic": str}` → `{"topic": str, "facts": list[str]}`.

In [ ]:
researcher_app = FastAPI()


class ResearchRequest(BaseModel):
    topic: str


class ResearchResponse(BaseModel):
    topic: str
    facts: list[str]


FACTS_BY_TOPIC = {
    "octopuses": [
        "Octopuses have three hearts.",
        "They can change color in under a second.",
        "Each of their arms has its own neural cluster.",
    ],
    "rome": [
        "Rome was founded in 753 BCE according to tradition.",
        "The Roman Empire at its peak spanned roughly 5 million km².",
        "Roman concrete used volcanic ash and is still studied today.",
    ],
}


@researcher_app.post("/research", response_model=ResearchResponse)
def research(req: ResearchRequest) -> ResearchResponse:
    facts = FACTS_BY_TOPIC.get(req.topic.lower())
    if facts is None:
        raise HTTPException(status_code=404, detail=f"No facts on file for {req.topic!r}")
    return ResearchResponse(topic=req.topic, facts=facts)


researcher_server = run_server_in_thread(researcher_app, port=8010)
print("Researcher running on http://127.0.0.1:8010")

In [ ]:
resp = httpx.post("http://127.0.0.1:8010/research", json={"topic": "octopuses"})
print(resp.status_code)
print(resp.json())